In [ ]:
"""
Project: Relational Online Food Delivery & Order Fulfilment Dashboard
Reg ID: 25SSDS415042
Course code: DSC202A

Description:
    A single-file Python application (SQLite + tkinter) that manages
    restaurants, delivery orders and order items.

Features:
    - Three related tables with foreign keys and ON DELETE CASCADE
    - Auto-seeding of sample data when the database is empty
    - Insert panel: finds an existing restaurant or creates a new one,
      then adds one order and one order item
    - Delete panel with dynamic dropdowns (restaurant / order / item)
    - Four tabs: combined JOIN view and the three raw tables
    - Input validation (no blanks, price must be a non-negative number)

How to run:
    python filename.py
    (the database file food_delivery.db is created automatically)
"""


import sqlite3
import math
from datetime import date
import tkinter as tk
from tkinter import ttk, messagebox

DB_FILE = "food_delivery.db"


# 1. DATABASE LAYER
def get_connection():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")  # needed on EVERY connection
    return conn


def create_tables():
    conn = get_connection()
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS restaurants (
            id    INTEGER PRIMARY KEY AUTOINCREMENT,
            name  TEXT NOT NULL UNIQUE COLLATE NOCASE,
            email TEXT NOT NULL DEFAULT ''
        );

        CREATE TABLE IF NOT EXISTS delivery_orders (
            id               INTEGER PRIMARY KEY AUTOINCREMENT,
            restaurant_id    INTEGER NOT NULL,
            delivery_address TEXT NOT NULL,
            order_date       TEXT NOT NULL,
            FOREIGN KEY (restaurant_id) REFERENCES restaurants(id)
                ON DELETE CASCADE
        );

        CREATE TABLE IF NOT EXISTS order_items (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            order_id   INTEGER NOT NULL,
            dish_name  TEXT NOT NULL,
            dish_price REAL NOT NULL,
            FOREIGN KEY (order_id) REFERENCES delivery_orders(id)
                ON DELETE CASCADE
        );
    """)
    conn.commit()
    conn.close()


def seed_if_empty():
    conn = get_connection()
    count = conn.execute("SELECT COUNT(*) FROM restaurants").fetchone()[0]
    if count == 0:
        conn.executemany(
            "INSERT INTO restaurants (name, email) VALUES (?, ?)",
            [("Pizza Palace", "hello@pizzapalace.com"),
             ("Biryani House", "orders@biryanihouse.com"),
             ("Dosa Corner", "contact@dosacorner.com")])
        conn.executemany(
            "INSERT INTO delivery_orders (restaurant_id, delivery_address, order_date) "
            "VALUES (?, ?, ?)",
            [(1, "12 MG Road, Apt 4B", "2026-10-30"),
             (1, "45 Indiranagar, 2nd Cross", "2026-10-31"),
             (2, "88 Koramangala, 5th Block", "2026-11-01"),
             (2, "7 Jayanagar, 4th Block", "2026-11-02"),
             (3, "21 HSR Layout, Sector 2", "2026-11-03")])
        conn.executemany(
            "INSERT INTO order_items (order_id, dish_name, dish_price) VALUES (?, ?, ?)",
            [(1, "Margherita Pizza", 349.0),
             (1, "Garlic Bread", 129.0),
             (2, "Pepperoni Pizza", 429.0),
             (3, "Chicken Biryani", 299.0),
             (4, "Mutton Biryani", 399.0),
             (4, "Raita", 49.0),
             (5, "Masala Dosa", 99.0),
             (5, "Filter Coffee", 40.0)])
        conn.commit()
    conn.close()


# 2. HELPER FUNCTIONS
def add_record(restaurant, address, dish, price):
    """Find or create the restaurant, then create 1 order + 1 item."""
    conn = get_connection()
    try:
        row = conn.execute(
            "SELECT id FROM restaurants WHERE name = ? COLLATE NOCASE",
            (restaurant,)).fetchone()
        if row:
            restaurant_id = row[0]
        else:
            cur = conn.execute(
                "INSERT INTO restaurants (name, email) VALUES (?, '')", (restaurant,))
            restaurant_id = cur.lastrowid

        cur = conn.execute(
            "INSERT INTO delivery_orders (restaurant_id, delivery_address, order_date) "
            "VALUES (?, ?, ?)",
            (restaurant_id, address, date.today().isoformat()))
        order_id = cur.lastrowid

        conn.execute(
            "INSERT INTO order_items (order_id, dish_name, dish_price) VALUES (?, ?, ?)",
            (order_id, dish, price))
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def delete_row(table, row_id):
    """table is chosen by our code (never by user text), so this is safe."""
    conn = get_connection()
    try:
        conn.execute(f"DELETE FROM {table} WHERE id = ?", (row_id,))
        conn.commit()
    finally:
        conn.close()


def fetch_all(query):
    conn = get_connection()
    cur = conn.execute(query)
    rows = cur.fetchall()
    headers = [d[0] for d in cur.description]
    conn.close()
    return headers, rows


JOIN_QUERY = """
    SELECT o.id AS "Order ID",
           r.name AS "Restaurant Name",
           o.delivery_address AS "Delivery Address",
           o.order_date AS "Order Date",
           i.dish_name AS "Dish Name",
           i.dish_price AS "Dish Price"
    FROM delivery_orders o
    JOIN restaurants r ON r.id = o.restaurant_id
    LEFT JOIN order_items i ON i.order_id = o.id
    ORDER BY o.id, i.id
"""

DELETE_QUERIES = {
    "Restaurant": ("restaurants",
                   "SELECT id, 'Restaurant #' || id || ' - ' || name FROM restaurants"),
    "Delivery Order": ("delivery_orders",
                       "SELECT o.id, 'Order #' || o.id || ' - ' || r.name || ' - ' || o.delivery_address "
                       "FROM delivery_orders o JOIN restaurants r ON r.id = o.restaurant_id"),
    "Order Item": ("order_items",
                   "SELECT id, 'Item #' || id || ' - ' || dish_name || ' (Order #' || order_id || ')' "
                   "FROM order_items"),
}


# 3. GUI
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Food Delivery & Order Fulfilment Dashboard")
        self.geometry("950x650")

        self.target_map = {}  # dropdown label -> database id

        self.build_insert_panel()
        self.build_delete_panel()
        self.build_tabs()
        self.refresh_all()

    # ----- Insert panel -----
    def build_insert_panel(self):
        frame = ttk.LabelFrame(self, text="Insert Panel")
        frame.pack(fill="x", padx=10, pady=5)

        self.var_restaurant = tk.StringVar()
        self.var_address = tk.StringVar()
        self.var_dish = tk.StringVar()
        self.var_price = tk.StringVar()

        fields = [("Restaurant Name", self.var_restaurant),
                  ("Delivery Address", self.var_address),
                  ("Dish Name", self.var_dish),
                  ("Dish Price", self.var_price)]
        for i, (label, var) in enumerate(fields):
            ttk.Label(frame, text=label).grid(row=i // 2, column=(i % 2) * 2,
                                              padx=5, pady=5, sticky="e")
            ttk.Entry(frame, textvariable=var, width=30).grid(
                row=i // 2, column=(i % 2) * 2 + 1, padx=5, pady=5)

        ttk.Button(frame, text="Add Record", command=self.on_add).grid(
            row=2, column=0, columnspan=4, pady=5)

    def on_add(self):
        restaurant = self.var_restaurant.get().strip()
        address = self.var_address.get().strip()
        dish = self.var_dish.get().strip()
        price_text = self.var_price.get().strip()

        if not restaurant or not address or not dish or not price_text:
            messagebox.showerror("Validation Error", "All fields are required.")
            return
        try:
            price = float(price_text)
        except ValueError:
            messagebox.showerror("Validation Error", "Dish price must be a number.")
            return
        if math.isnan(price) or math.isinf(price) or price < 0:
            messagebox.showerror("Validation Error",
                                 "Dish price must be a non-negative number.")
            return

        try:
            add_record(restaurant, address, dish, price)
        except sqlite3.Error as e:
            messagebox.showerror("Database Error", str(e))
            return

        # clear the text variables, then refresh everything
        for var in (self.var_restaurant, self.var_address, self.var_dish, self.var_price):
            var.set("")
        self.refresh_all()

    # ----- Delete panel -----
    def build_delete_panel(self):
        frame = ttk.LabelFrame(self, text="Delete Panel")
        frame.pack(fill="x", padx=10, pady=5)

        self.var_delete_type = tk.StringVar(value="Restaurant")
        self.var_delete_target = tk.StringVar()

        ttk.Label(frame, text="Delete a:").grid(row=0, column=0, padx=5, pady=5)
        type_box = ttk.Combobox(frame, textvariable=self.var_delete_type,
                                values=list(DELETE_QUERIES.keys()),
                                state="readonly", width=18)
        type_box.grid(row=0, column=1, padx=5, pady=5)
        type_box.bind("<<ComboboxSelected>>", lambda e: self.refresh_delete_targets())

        self.target_box = ttk.Combobox(frame, textvariable=self.var_delete_target,
                                       state="readonly", width=60)
        self.target_box.grid(row=0, column=2, padx=5, pady=5)

        ttk.Button(frame, text="Delete Selected", command=self.on_delete).grid(
            row=0, column=3, padx=5, pady=5)

    def refresh_delete_targets(self):
        """Rebuild dropdown from the database so it never holds a stale ID."""
        _, query = DELETE_QUERIES[self.var_delete_type.get()]
        _, rows = fetch_all(query)
        self.target_map = {label: row_id for row_id, label in rows}
        self.target_box["values"] = list(self.target_map.keys())
        self.var_delete_target.set("")  # clear any old selection

    def on_delete(self):
        label = self.var_delete_target.get()
        if label not in self.target_map:
            messagebox.showerror("Selection Error", "Please select something to delete.")
            return
        table, _ = DELETE_QUERIES[self.var_delete_type.get()]
        try:
            delete_row(table, self.target_map[label])
        except sqlite3.Error as e:
            messagebox.showerror("Database Error", str(e))
            return
        self.refresh_all()

    # ----- Tabs -----
    def build_tabs(self):
        self.notebook = ttk.Notebook(self)
        self.notebook.pack(fill="both", expand=True, padx=10, pady=5)

        self.tabs = {}  # name -> (Treeview, query)
        tab_defs = [("Combined View", JOIN_QUERY),
                    ("Restaurants", "SELECT * FROM restaurants"),
                    ("Delivery Orders", "SELECT * FROM delivery_orders"),
                    ("Order Items", "SELECT * FROM order_items")]
        for name, query in tab_defs:
            frame = ttk.Frame(self.notebook)
            self.notebook.add(frame, text=name)
            tree = ttk.Treeview(frame, show="headings")
            scroll = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
            tree.configure(yscrollcommand=scroll.set)
            tree.pack(side="left", fill="both", expand=True)
            scroll.pack(side="right", fill="y")
            self.tabs[name] = (tree, query)

    def refresh_tabs(self):
        for tree, query in self.tabs.values():
            headers, rows = fetch_all(query)
            tree.delete(*tree.get_children())
            tree["columns"] = headers
            for h in headers:
                tree.heading(h, text=h)
                tree.column(h, width=130, anchor="w")
            for row in rows:
                tree.insert("", "end", values=row)

    def refresh_all(self):
        self.refresh_tabs()
        self.refresh_delete_targets()



# 4. MAIN
if __name__ == "__main__":
    create_tables()
    seed_if_empty()
    App().mainloop()